# ESG RAG — Evaluation Notebook
Compare **Basic**, **HyDE**, and **MQR** retrieval strategies using sample ESG questions.

In [ ]:
import sys
sys.path.insert(0, '..')

import fitz
import numpy as np
import pandas as pd

from rag.ingestion import load_pdf, chunk_text
from rag.embeddings import get_embeddings
from rag.vectorstore import VectorStore
from rag.retriever import basic_retrieve, hyde_retrieve, mqr_retrieve
from rag.llm import generate_answer

print('Imports OK')

## 1. Load a PDF and build the vector store

In [ ]:
PDF_PATH = 'your_esg_report.pdf'  # <-- change to your PDF

with open(PDF_PATH, 'rb') as f:
    pages = load_pdf(f.read(), PDF_PATH)

chunks = chunk_text(pages)
print(f'Pages: {len(pages)}  |  Chunks: {len(chunks)}')

In [ ]:
store = VectorStore()
embeddings = get_embeddings([c['text'] for c in chunks])
store.add(chunks, embeddings)
print(f'Indexed {len(store)} chunks')

## 2. Define evaluation questions

In [ ]:
questions = [
    'What are the company carbon emission reduction targets?',
    'How does the company manage water consumption?',
    'What governance policies exist for ESG oversight?',
    'What social initiatives does the company have for employees?',
    'What is the net zero commitment timeline?',
]

## 3. Run all three retrieval strategies

In [ ]:
results = []

for q in questions:
    print(f'Q: {q}')
    for method_name, fn in [('Basic', basic_retrieve), ('HyDE', hyde_retrieve), ('MQR', mqr_retrieve)]:
        chunks_retrieved = fn(q, store, k=5)
        answer = generate_answer(q, chunks_retrieved)
        avg_score = np.mean([c['score'] for c in chunks_retrieved])
        results.append({
            'question': q,
            'method': method_name,
            'answer': answer,
            'avg_retrieval_score': round(avg_score, 4),
            'sources': [f"{c['source']} p.{c['page']}" for c in chunks_retrieved],
        })
    print()

df = pd.DataFrame(results)
print('Done!')

## 4. Compare retrieval scores by method

In [ ]:
import matplotlib.pyplot as plt

pivot = df.pivot_table(index='question', columns='method', values='avg_retrieval_score')
pivot.plot(kind='bar', figsize=(12, 5), title='Average Retrieval Score by Method')
plt.ylabel('Cosine Similarity')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(pivot.mean().to_string())

## 5. Browse answers

In [ ]:
pd.set_option('display.max_colwidth', 300)
df[['question', 'method', 'avg_retrieval_score', 'answer']]